# Decompressing a Sentinel-2 scene: GeoTIFF vs Rumi

Same pixels, same machine, decompress only. The encoded bytes are loaded into RAM
before timing starts, so no page cache or GDAL block cache is in the measurement.

In [ ]:
!pip install -q "rumi-eo[write]" rasterio

In [ ]:
import os
os.environ["GDAL_CACHEMAX"] = "64"
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"

import hashlib, statistics as st, time
import numpy as np, rasterio, geozl, rumi

THREADS = os.cpu_count()
print(f"rumi {rumi.__version__}  geozl {geozl.__version__}  GDAL {rasterio.__gdal_version__}  {THREADS} cores")

## The scene

S2A_37MBV_20241029, true-colour, 318 MB on S3.

In [ ]:
URL = ("https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/"
       "37/M/BV/2024/10/S2A_37MBV_20241029_0_L2A/TCI.tif")
!wget -q -O TCI.tif {URL}

with rasterio.open("TCI.tif") as ds:
    print(ds.count, "x", ds.shape, ds.dtypes[0])
    print("blocks    ", ds.block_shapes[0])
    print("compress  ", ds.profile["compress"], "predictor", ds.tags(ns="IMAGE_STRUCTURE")["PREDICTOR"])
    print("interleave", ds.profile["interleave"])
    arr = ds.read()

RAW_MB = arr.nbytes / 1e6
print(f"\nuncompressed {RAW_MB:.1f} MB")

## Write it as Rumi

One frame per 1024x1024 tile holding all three bands, compressed with a planar
predictor, zigzag, and PFOR bit-packing.

In [ ]:
TILE = 1024

frames = rumi.frames(arr, "b (row h) (col w) -> row col (b h w)", tile_size=TILE)
graphs = {}
t0 = time.perf_counter()
for f in frames:
    g = graphs.get(f.data.shape)
    if g is None:
        g = graphs[f.data.shape] = geozl.graph(f.data, "planar>zigzag>pfor")
    f.compressed = geozl.compress(f.data, graph=g)
encode_s = time.perf_counter() - t0

_, header = rumi.write("TCI.rumi", frames)

tif_mb = os.path.getsize("TCI.tif") / 1e6
rumi_mb = os.path.getsize("TCI.rumi") / 1e6
print(f"{len(frames)} frames encoded in {encode_s:.1f}s")
print(f"GeoTIFF  {tif_mb:6.1f} MB")
print(f"Rumi     {rumi_mb:6.1f} MB   ({100 * (tif_mb - rumi_mb) / tif_mb:.1f}% smaller)")

## Benchmark

Both files are read into RAM first. Each engine decodes five times, we keep the median.

In [ ]:
tif_bytes = open("TCI.tif", "rb").read()
rumi_bytes = open("TCI.rumi", "rb").read()

def timeit(fn, n=5):
    times, digest = [], None
    for _ in range(n):
        t0 = time.perf_counter()
        out = np.asarray(fn())
        times.append(time.perf_counter() - t0)
        if digest is None:
            digest = hashlib.sha256(np.ascontiguousarray(out)).hexdigest()[:12]
        del out
    return st.median(times), digest

def gdal(driver, threads):
    mf = rasterio.MemoryFile(tif_bytes)
    ds = mf.open(driver=driver, NUM_THREADS=str(threads))
    return timeit(ds.read)

def rumi_read(threads):
    rumi.set_num_threads(threads)
    return timeit(lambda: rumi.read(rumi_bytes, header))

In [ ]:
results = {}
for threads in (1, THREADS):
    for label, run in (("GDAL GTiff", lambda t: gdal("GTiff", t)),
                       ("GDAL LIBERTIFF", lambda t: gdal("LIBERTIFF", t)),
                       ("Rumi", rumi_read)):
        os.environ["GDAL_NUM_THREADS"] = str(threads)
        results[label, threads] = run(threads)

print(f"{'':16s} {'threads':>7} {'ms':>8} {'MB/s':>8}")
for (label, threads), (secs, _) in results.items():
    print(f"{label:16s} {threads:7d} {secs * 1000:8.1f} {RAW_MB / secs:8.0f}")

assert len({d for _, d in results.values()}) == 1, "engines disagree"
print("\nall engines produced identical pixels")

In [ ]:
for threads in (1, THREADS):
    r = results["Rumi", threads][0]
    for other in ("GDAL GTiff", "GDAL LIBERTIFF"):
        print(f"{threads:2d} threads: Rumi is {results[other, threads][0] / r:5.2f}x faster than {other}")

## Caveats

The GeoTIFF is pixel-interleaved and DEFLATE-compressed; the Rumi file stores each
tile planar and skips entropy coding. Part of the gap is the codec and part is the
layout. Colab gives you 2 vCPUs, so the multi-threaded column is much less
interesting than on a real machine.

Same notebook on an Apple M5 (4 performance + 6 efficiency cores):

| | 1 thread | 10 threads |
|---|---:|---:|
| GDAL GTiff | 1350 ms | 196 ms |
| GDAL LIBERTIFF | 720 ms | 103 ms |
| Rumi | 122 ms | 39 ms |
